# Model Training

Train baseline classifiers on the per-subject dataset created in the previous notebook.
We start with Subject 01 and compare SVM (RBF), kNN, and MLP.


## What you will do
- Load the precomputed feature dataset for Subject 01.
- Train three baseline classifiers (SVM, kNN, MLP).
- Report test accuracy for each model.

In [1]:
from pathlib import Path
import pickle

from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

## Load Subject 01 dataset
This file is generated in the windowing and feature extraction notebook.


In [2]:
DATA_PATH = Path("../data/processed/db1_subject01_win20_hop1.pkl")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing dataset: {DATA_PATH}. Run notebook 03 to generate it."
    )

with DATA_PATH.open("rb") as f:
    dataset = pickle.load(f)

X_train = dataset["X_train"]
y_train = dataset["y_train"]
X_test = dataset["X_test"]
y_test = dataset["y_test"]

X_train.shape, X_test.shape


((25386, 30), (216565, 30))

## Train and evaluate baseline models
The features are already standardized in the saved dataset, so we can train directly.


In [3]:
models = {
    # "SVM (RBF)": SVC(kernel="rbf", C=1.0, gamma="scale"),
    # "kNN": KNeighborsClassifier(n_neighbors=5),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=2000,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=42,
    ),
    "RF": RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2,
    ),
}

MODEL_DIR = Path("../data/processed/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

subject_id = int(dataset["meta"]["subject_id"])
win_samples = int(dataset["meta"]["win_samples"])
hop_samples = int(dataset["meta"]["hop_samples"])


def _model_id(name: str) -> str:
    return name.lower().replace(" ", "_").replace("(", "").replace(")", "")


for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: accuracy = {acc:.4f}")

    model_path = (
        MODEL_DIR
        / f"db1_subject{subject_id:02d}_{_model_id(name)}_win{win_samples}_hop{hop_samples}.pkl"
    )
    with model_path.open("wb") as f:
        pickle.dump(model, f)
    print(f"Saved {model_path}")


MLP: accuracy = 0.8150
Saved ../data/processed/models/db1_subject01_mlp_win20_hop1.pkl
RF: accuracy = 0.8167
Saved ../data/processed/models/db1_subject01_rf_win20_hop1.pkl


## Interpretation and next step
Overall accuracy gives a quick sense of how well each model matches the ground-truth labels on the test windows.
Because rest dominates the dataset, accuracy can look strong even when some movements are confused.
Proceed to Notebook 05 for evaluation and a deeper error analysis.
